# ERA5 Land Data
ERA5 data extraction notebook.


In [ ]:
import ee
import os
import pandas as pd
import matplotlib.pyplot as plt
import time
import datetime
import re
import json
print('Libraries loaded.')


In [ ]:
# Authenticate and initialize ----------------------------------------------------------------------
ee.Authenticate()
ee.Initialize(project='fire-seasons')

## Configurations

In [ ]:
# Imports and configurations -----------------------------------------------------------------------

YEARS = range(2002, 2026) # 2002 because of an onset-90d possibility
SCALE = 11132
ERA5_COLLECTION = 'ECMWF/ERA5_LAND/DAILY_AGGR'
BANDS = ['temperature_2m', 'total_precipitation_sum']
RUN_ID = datetime.datetime.now().strftime('%Y%m%d_%H%M')

# Output paths and directories ---------------------------------------------------------------------

BASE_OUT_DIR = r'C:\Users\ibekar\Documents\GitProjects\TGPF'  # Windows
# BASE_OUT_DIR = '/Users/ibekar/Github/TGPF'                  # Mac

DRIVE_FOLDER = "ERA5_global_raw"
GEO_PATH = r'C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\eco_geometries.json'

save_dir = os.path.join(BASE_OUT_DIR, 'inputs', 'raw_data', 'era5')

print()
print(f'Base output directory: {BASE_OUT_DIR}')
print(f'Drive save output: {DRIVE_FOLDER}')
print(f'Geo file directory: {GEO_PATH}')
print(f'save directory: {save_dir}')

### Regional data load from GEE

In [ ]:
# # LOAD MEDITERRANEAN BASIN ECOREGIONS --------------------------------------------------------------
# med_bbox = ee.Geometry.BBox(-10, 28, 42, 48)

# ecoregions_med = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017") #.filterBounds(med_bbox)

# n_eco    = ecoregions_med.size().getInfo()
# eco_list = ecoregions_med.select(['ECO_ID', 'ECO_NAME', 'BIOME_NUM', 'BIOME_NAME']).getInfo()

# print(f'Number of ecoregions intersecting bounding box: {n_eco}')
# for f in eco_list['features']:
#     p = f['properties']
#     print(p['ECO_ID'], '|', p['ECO_NAME'], '|', p['BIOME_NAME'])

# # BUILD ECO RECORDS --------------------------------------------------------------------------------
# eco_records = []
# for f in eco_list['features']:
#     p = f['properties']
#     eco_records.append({
#         'eco_id'    : p['ECO_ID'],
#         'eco_name'  : p['ECO_NAME'],
#         'biome_num' : p['BIOME_NUM'],
#         'biome_name': p['BIOME_NAME'],
#         'geometry'  : ee.Geometry(f['geometry'])
#     })

# print(f'Built {len(eco_records)} ecoregion records.')

# # SUBSETTING (set to None to disable) --------------------------------------------------------------
# TEST_N   = None
# TEST_IDS = None

# eco_run = eco_records

# if TEST_IDS is not None:
#     eco_run = [e for e in eco_run if e['eco_id'] in TEST_IDS]
#     print(f'Subsetting to {len(eco_run)} ecoregions by ID: {TEST_IDS}')

# if TEST_N is not None:
#     eco_run = eco_run[:TEST_N]
#     print(f'Subsetting to first {TEST_N} ecoregions.')

# print(f'Running on {len(eco_run)} / {len(eco_records)} ecoregions.')

### Global ecoregion load from disk

In [ ]:
# LOAD GLOBAL ECOREGIONS FROM GEE (server-side geometries) ----------------------------------------

# Load the full FeatureCollection on GEE's side - geometry stays on the server
ecoregions_fc = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")

# Pull only the lightweight properties to local Python - paginated to avoid response size limit
eco_list_features = []
batch_size = 50
n_eco = ecoregions_fc.size().getInfo()
print(f'Total ecoregions in GEE: {n_eco}')

for start in range(0, n_eco, batch_size):
    batch = ecoregions_fc.select(['ECO_ID', 'ECO_NAME', 'BIOME_NUM', 'BIOME_NAME']) \
                         .toList(batch_size, start) \
                         .getInfo()
    eco_list_features.extend(batch)
    print(f'  Fetched {len(eco_list_features)} / {n_eco}')

print(f'Fetched {len(eco_list_features)} ecoregion property records.')

# BUILD ECO RECORDS --------------------------------------------------------------------------------
# Geometry is looked up server-side by ECO_ID at extraction time - never serialized locally
eco_records = []
for f in eco_list_features:
    p = f['properties']
    eco_records.append({
        'eco_id'    : p['ECO_ID'],
        'eco_name'  : p['ECO_NAME'],
        'biome_num' : p['BIOME_NUM'],
        'biome_name': p['BIOME_NAME'],
    })

print(f'Built {len(eco_records)} ecoregion records.')

In [ ]:
# SUBSETTING (set to None to disable) --------------------------------------------------------------
TEST_N   = None
TEST_IDS = None

eco_run = eco_records

if TEST_IDS is not None:
    eco_run = [e for e in eco_run if e['eco_id'] in TEST_IDS]
    print(f'Subsetting to {len(eco_run)} ecoregions by ID: {TEST_IDS}')

if TEST_N is not None:
    eco_run = eco_run[:TEST_N]
    print(f'Subsetting to first {TEST_N} ecoregions.')

print(f'Running on {len(eco_run)} / {len(eco_records)} ecoregions.')

## Helper function
ERA5 extraction for ecoregions

In [47]:
def extract_era5_for_ecoregion(eco_feature):
    '''Extract ERA5 data for a single eco-region feature.'''
    eco_id = eco_feature['eco_id']
    eco_name = eco_feature['eco_name']

    # Server-side geometry lookup by ECO_ID - never serialized locally
    geometry_simplified = (ecoregions_fc
                           .filter(ee.Filter.eq('ECO_ID', eco_id))
                           .first()
                           .geometry()
                           .simplify(5000))

    # Filter ERA5 collection by date
    era5 = (ee.ImageCollection(ERA5_COLLECTION)
            .filterDate(f'{YEARS[0]}-01-01', f'{YEARS[-1]+1}-01-01')
            .select(BANDS))

    def image_to_feature(image):
        stats = image.reduceRegion(
            reducer   = ee.Reducer.mean(),
            geometry  = geometry_simplified,
            scale     = SCALE,
            maxPixels = 1e9,
            bestEffort= True
        )
        return ee.Feature(None, {
            "eco_id"        : eco_id,
            "date"          : image.date().format('YYYY-MM-dd'),
            "eco_name"      : eco_name,
            "temperature_K" : stats.get('temperature_2m'),
            "precip_m"      : stats.get('total_precipitation_sum')
        })

    daily_fc = ee.FeatureCollection(era5.map(image_to_feature))
    return daily_fc

## Submission loop

In [48]:
n_submitted = 0

for e in eco_run:
    task_desc = f'ERA5_{RUN_ID}_eco_{e["eco_id"]}'
    file_name = f'ERA5_eco{e["eco_id"]}_{re.sub(r"[^a-zA-Z0-9]", "_", e["eco_name"])}'
    print(f'Submitting {task_desc}...')

    daily_fc = extract_era5_for_ecoregion(e)

    task = ee.batch.Export.table.toDrive(
        collection    = daily_fc,
        description   = task_desc,
        folder        = DRIVE_FOLDER,
        fileNamePrefix= file_name,
        fileFormat    = 'CSV',
        selectors     = ['eco_id', 'eco_name', 'date', 'temperature_K', 'precip_m']
    )
    task.start()
    print(f' {n_submitted + 1} → Submitted.')
    n_submitted += 1

print(f'All {n_submitted} tasks submitted.')
print('Monitor at: https://code.earthengine.google.com/tasks')

Submitting ERA5_20260424_1338_eco_0...
 1 → Submitted.
Submitting ERA5_20260424_1338_eco_111...
 2 → Submitted.
Submitting ERA5_20260424_1338_eco_321...
 3 → Submitted.
Submitting ERA5_20260424_1338_eco_322...
 4 → Submitted.
Submitting ERA5_20260424_1338_eco_616...
 5 → Submitted.
Submitting ERA5_20260424_1338_eco_112...
 6 → Submitted.
Submitting ERA5_20260424_1338_eco_113...
 7 → Submitted.
Submitting ERA5_20260424_1338_eco_116...
 8 → Submitted.
Submitting ERA5_20260424_1338_eco_323...
 9 → Submitted.
Submitting ERA5_20260424_1338_eco_114...
 10 → Submitted.
Submitting ERA5_20260424_1338_eco_319...
 11 → Submitted.
Submitting ERA5_20260424_1338_eco_309...
 12 → Submitted.
Submitting ERA5_20260424_1338_eco_696...
 13 → Submitted.
Submitting ERA5_20260424_1338_eco_697...
 14 → Submitted.
Submitting ERA5_20260424_1338_eco_701...
 15 → Submitted.
Submitting ERA5_20260424_1338_eco_702...
 16 → Submitted.
Submitting ERA5_20260424_1338_eco_704...
 17 → Submitted.
Submitting ERA5_20260424_

In [50]:
print('Monitoring tasks...')

tasks     = ee.data.getTaskList()
my_tasks = [t for t in tasks if t["description"].startswith(f'ERA5_{RUN_ID}')]
running   = sum(1 for t in my_tasks if t['state'] == 'RUNNING')
completed = sum(1 for t in my_tasks if t['state'] == 'COMPLETED')
failed    = sum(1 for t in my_tasks if t['state'] == 'FAILED')
ready     = sum(1 for t in my_tasks if t['state'] == 'READY')

print(f' READY: {ready} | RUNNING: {running} | COMPLETED: {completed} | FAILED: {failed}')

if failed > 0:
    print(f'\nWARNING: {failed} tasks failed:')
    for t in my_tasks:
        if t['state'] == 'FAILED':
            print(f'  - {t["description"]}')
            print(f'    {t["error_message"]}')

Monitoring tasks...
 READY: 845 | RUNNING: 1 | COMPLETED: 2 | FAILED: 1

  - ERA5_20260424_1338_eco_0
    Unable to transform geometry into projection EPSG:4326: affine [0.10000045742818514, 0.0, -180.05, 0.0, -0.10000045742818514, 90.05].


In [ ]:
PLOT_ECO_ID = 701  # change this to inspect any ecoregion

plot_path = os.path.join(save_dir, f'ERA5_eco_{PLOT_ECO_ID}.csv')
out = pd.read_csv(plot_path)

out["date"] = pd.to_datetime(out["date"])


plt.figure(figsize=(12, 6))
plt.suptitle(f'ERA5 Daily Data for Ecoregion {out['eco_name'].iloc[0]}', fontsize=16)
plt.subplot(2, 1, 1)
plt.plot(out["date"], out["precip_m"])
plt.title("Daily Precipitation (m)")
plt.subplot(2, 1, 2)
plt.plot(out["date"], out["temperature_K"])
plt.title("Daily Temperature (K)")
plt.xlabel("Date")
plt.ylabel("Value")
plt.tight_layout()
plt.show()
